Samajhna pehle: Kal ek model ka ek run track kiya tha. Aaj poora comparison track hoga:

- Har model family (LogReg, Tree, RandomForest, XGBoost) ka parent run.
- Grid search ke har hyperparameter combination ka child (nested) run. UI mein parent ke neeche khulte hain, to “kaunsa max_depth kitna accha tha” ka poora nakshaa ek jagah dikhta hai.
- Har family ka finalist run, jisme model khud (log_model) bhi save hota hai, taaki “yeh score is model se aaya tha” ka saboot rahe.

Aaj ke rules (kal wale hi):

1. Sirf CV metrics. Test set ka koi number tracking mein nahi jaayega (guard code mein hai).
2. Yeh after-the-fact record hai. Day 83-84 ke experiments pehle ho chuke hain, aur final model ka faisla reports/final_model_choice_day84.md mein hai. Aaj ka tracking use badalta nahi.
3. mlflow_store/ Git mein nahi jaayega. Jo numbers zaroori hain unki CSV reports/ mein bhi save hogi.

1: Imports

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parents[1]))
import warnings; warnings.filterwarnings('ignore')
import json, re, time

import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold

from src.models.search_spaces import get_search_spaces, pick_one_se
from src.tracking.experiments import log_grid_children, track_cv_run
from src.tracking.mlflow_utils import git_info, log_cv_metrics, setup_tracking

2: Data, same folds, tracking

In [2]:
X = pd.read_csv('../../data/processed/X_train_v2.csv').astype(float)
y = pd.read_csv('../../data/processed/y_train.csv').squeeze()
assert len(X) == len(y)

spaces = get_search_spaces((y == 0).sum() / (y == 1).sum())
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)            # Day 64B se wahi folds
SCORING = {'roc_auc': 'roc_auc', 'ap': 'average_precision', 'recall': 'recall', 'f1': 'f1'}

EXPERIMENT = 'churn-model-comparison'     # notebook dobara chalana ho to naam badlo (v2), warna duplicate runs bante hain
setup_tracking(EXPERIMENT)

'2'

3: Baseline

In [3]:
base = track_cv_run('baseline_prior', DummyClassifier(strategy='prior'), X, y, cv, SCORING,
                    tags={'purpose': 'baseline'})
print({k: round(v, 3) for k, v in base['summary'].items() if k.endswith('mean')})

{'cv_roc_auc_mean': 0.5, 'cv_ap_mean': 0.566, 'cv_recall_mean': 1.0, 'cv_f1_mean': 0.723, 'cv_fit_time_mean': 0.002}


AUC 0.5 aani chahiye. Sab models isi se compare honge.

4: Har family ka grid, parent aur child runs ke saath

In [4]:
FAMILIES = ['LogReg', 'Tree', 'RandomForest', 'XGBoost']      # laptop slow ho to RF/XGBoost hata do
searched = {}
for fam in FAMILIES:
    est, grid = spaces[fam]
    n_combo = int(np.prod([len(v) for v in grid.values()]))
    t0 = time.time()
    with mlflow.start_run(run_name=f'grid_{fam}'):
        mlflow.log_params({'grid_size': n_combo,
                           'grid': json.dumps({k: [str(v) for v in vs] for k, vs in grid.items()})})
        mlflow.set_tags({**git_info(), 'stage': 'cv_grid_parent', 'family': fam, 'test_set_used': 'false'})
        gs = GridSearchCV(est, grid, cv=cv, scoring=SCORING, refit='roc_auc', n_jobs=-1).fit(X, y)
        log_grid_children(gs, list(SCORING), tags={'family': fam})
        b = gs.best_index_
        log_cv_metrics({'cv_best_roc_auc_mean': float(gs.cv_results_['mean_test_roc_auc'][b]),
                        'cv_best_roc_auc_std': float(gs.cv_results_['std_test_roc_auc'][b]),
                        'search_minutes': (time.time() - t0) / 60})
    searched[fam] = gs
    print(f'{fam}: {n_combo} combos, {(time.time() - t0) / 60:.1f} min')

LogReg: 4 combos, 0.1 min
Tree: 12 combos, 0.0 min
RandomForest: 18 combos, 0.5 min
XGBoost: 24 combos, 0.2 min


Time lagbhag Day 84 ke search step jitna lagega (nested CV nahi hai, wo sabse slow tha). Grids wahi hain jo Day 84 mein the.

5: Sanity check: re-run ka best score Day 84 se match karta hai?

In [5]:
day84 = pd.read_csv('../../reports/tuning_results_day84.csv', index_col=0)
chk = pd.DataFrame({
    'rerun_best_cv_auc': {f: gs.cv_results_['mean_test_roc_auc'][gs.best_index_] for f, gs in searched.items()},
    'day84_best_cv_auc': {f: day84.loc[f, 'best_cv_auc'] for f in searched},
})
chk['abs_diff'] = (chk['rerun_best_cv_auc'] - chk['day84_best_cv_auc']).abs()
print(chk.round(4))
assert (chk['abs_diff'] < 1e-3).all(), 'Day 84 se fark: folds, features ya grid alag hain'

              rerun_best_cv_auc  day84_best_cv_auc  abs_diff
LogReg                   0.7961             0.7961       0.0
Tree                     0.7850             0.7850       0.0
RandomForest             0.7988             0.7988       0.0
XGBoost                  0.8020             0.8020       0.0


Agar assert fail ho, to tracking mein nahi, usse pehle kuch alag hai (folds, X_train_v2.csv, ya grid). Pehle wo dhoondo.

6: Finalists (1-SE pick, model save ke saath)

In [7]:
finalists = {}
for fam, gs in searched.items():
    pick = pick_one_se(gs, fam)
    est = clone(spaces[fam][0]).set_params(**pick)
    if 'n_jobs' in est.get_params():
        est.set_params(n_jobs=-1)
    finalists[fam] = track_cv_run(
        f'finalist_{fam}', est, X, y, cv, SCORING, fit_and_log_model=False,
        tags={'stage': 'cv_finalist', 'family': fam,
              'selection_rule': '1-SE, deterministic complexity tie-break'})
    print(fam, pick, round(finalists[fam]['summary']['cv_roc_auc_mean'], 4))

LogReg {'m__C': 0.01} 0.7958
Tree {'max_depth': 3, 'min_samples_leaf': 60} 0.7753
RandomForest {'max_depth': 4, 'min_samples_leaf': 30, 'n_estimators': 200} 0.798
XGBoost {'learning_rate': 0.03, 'max_depth': 2, 'min_child_weight': 10, 'n_estimators': 150} 0.7988


7: Pre-registered choice se compare (sirf dekhne ke liye)

In [8]:
txt = open('../../reports/final_model_choice_day84.md').read()
prereg_fam = re.search(r'- Model: (.+)', txt).group(1).strip()
prereg_params = json.loads(re.search(r'- Params: (.+)', txt).group(1))
pick = pick_one_se(searched[prereg_fam], prereg_fam)
print('Pre-registered (Day 84):', prereg_fam, prereg_params)
print('Re-run 1-SE pick       :', pick)
print('Same params:', pick == prereg_params)

Pre-registered (Day 84): XGBoost {'learning_rate': 0.03, 'max_depth': 2, 'min_child_weight': 3, 'n_estimators': 150}
Re-run 1-SE pick       : {'learning_rate': 0.03, 'max_depth': 2, 'min_child_weight': 10, 'n_estimators': 150}
Same params: False


Agar False aaye, to ghabrao mat. Wajah wahi fit-time wala tie-break hai (jo upar bataya). Sach wohi file hai jo Day 84 mein commit hui, aur Day 85 ka test evaluation usi par hua. Is notebook se final model nahi badlega.

8: Comparison table, model round-trip, aur backup

In [11]:
runs = mlflow.search_runs(experiment_names=[EXPERIMENT], filter_string="tags.stage = 'cv_finalist'")
tbl = (runs[['tags.family', 'metrics.cv_roc_auc_mean', 'metrics.cv_roc_auc_std', 'metrics.cv_ap_mean',
             'metrics.cv_recall_mean', 'metrics.cv_fit_time_mean']]
       .set_index('tags.family').sort_values('metrics.cv_roc_auc_mean', ascending=False).round(4))
print(tbl)

for fam, out in finalists.items():
    if out.get('model_uri'):
        loaded = mlflow.sklearn.load_model(out['model_uri'])
        ok = np.allclose(loaded.predict_proba(X.head(100)), out['fitted'].predict_proba(X.head(100)))
        print(fam, '| model round-trip OK:', ok)
    else:
        print(fam, '| model round-trip OK: False (no model URI)')

tbl.to_csv('../../reports/mlflow_finalists_day92.csv')       # store Git mein nahi jaata, isliye backup

              metrics.cv_roc_auc_mean  metrics.cv_roc_auc_std  \
tags.family                                                     
XGBoost                        0.7988                  0.0160   
RandomForest                   0.7980                  0.0161   
LogReg                         0.7958                  0.0123   
LogReg                         0.7958                  0.0123   
LogReg                         0.7958                  0.0123   
LogReg                         0.7958                  0.0123   
Tree                           0.7753                  0.0164   
Tree                           0.7753                  0.0164   
Tree                           0.7753                  0.0164   

              metrics.cv_ap_mean  metrics.cv_recall_mean  \
tags.family                                                
XGBoost                   0.8218                  0.7514   
RandomForest              0.8219                  0.7606   
LogReg                    0.8203            

mlflow.sklearn.load_model bhi pickle load karta hai. Sirf apne banaye hue models load karo (Day 87 wali security baat).

#### 6. UI tour (mlflow ui --backend-store-uri sqlite:///mlflow_store/mlflow.db --port 5001)

1. Experiment churn-model-comparison kholo. grid_XGBoost ke aage ka ▶ dabao, aur 24 child runs khulenge.
2. Children ko cv_roc_auc_mean se sort karo. Top kitne runs ek-doosre ke std ke andar hain? Yahi plateau hai (Day 84 jaisa).
3. Children select karke Compare → parallel coordinates mein hp_max_depth, hp_learning_rate, cv_roc_auc_mean rakho. Kaunsa hyperparameter score ko sach mein hilata hai?
4. Filter: tags.stage = 'cv_finalist'. Chaaron finalists dikhenge.
5. Ek finalist kholo → Artifacts mein model/ folder (usme MLmodel aur requirements file). Yeh wahi cheez hai jo Day 87 ke manifest ne haath se karwaayi thi, ab MLflow khud likhta hai.
6. Filter: tags.test_set_used = 'false' → sab runs aane chahiye.

#### 7. Results ko imandaari se padho

- Finalists ke cv_roc_auc_mean ka fark unke cv_roc_auc_std se bada hai? Agar nahi, to “ek jeeta” mat bolo.
- Baseline (AUC 0.5) se sab kitne upar hain? Wahi asli signal hai.
- cv_fit_time_mean bhi dekho. Ek chhota gain agar 50 guna slow model se aaye, to production mein sawaal uthta hai.
- Tracking record karta hai, faisla nahi. Is table ko dekh kar tum final model badalna chahogi/chahoge to ruko. Final model ka faisla pehle ho chuka hai, aur test set Day 85 mein dekha ja chuka hai.